환경 확인하고 4개 csv 불러오기
-customers
-orders


In [1]:
# 환경 확인 - course_utils를 찾을 수 있게 프로젝트 루트를 검색 경로에 추가
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from course_utils.paths import get_project_root, get_data_dir
import pandas as pd

RAW = get_data_dir() / "raw"
print("프로젝트 루트:", get_project_root())
print("데이터 폴더:", RAW, RAW.exists())


프로젝트 루트: C:\dev\llm-data-analysis-course
데이터 폴더: C:\dev\llm-data-analysis-course\data\raw True


In [2]:
# customers, orders 불러오기
customers = pd.read_csv(RAW / "customers.csv")
orders    = pd.read_csv(RAW / "orders.csv")

print(customers.shape, orders.shape)


(150, 6) (300, 5)


In [3]:
# products, order_items 불러오기
products    = pd.read_csv(RAW / "products.csv")
order_items = pd.read_csv(RAW / "order_items.csv")

print(products.shape, order_items.shape)


(100, 4) (764, 5)


In [4]:
# oreders 의 상위 5개 row만 출력

print(orders.head())

   order_id  customer_id  order_date payment_method order_status
0         1          123  2026-07-08           card    completed
1         2           77  2025-09-23      naver_pay    cancelled
2         3          138  2026-01-20  bank_transfer    cancelled
3         4           57  2026-04-02      kakao_pay    cancelled
4         5          125  2026-02-21           card    cancelled


In [5]:
# 주문 상태의 종류는 몇 가지이고 각 상태별 주문 수량?
ret = orders["order_status"].value_counts(dropna=False)
print(type(ret))
print("상태 종류:", orders["order_status"].nunique(), "가지")
print(ret)

print()

# 결제 수단의 종류는 몇 가지이고 각 결제 수단별 주문 수량?
pay = orders["payment_method"].value_counts(dropna=False)
print("결제 수단 종류:", orders["payment_method"].nunique(), "가지")
print(pay)


<class 'pandas.Series'>
상태 종류: 3 가지
order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

결제 수단 종류: 4 가지
payment_method
kakao_pay        79
naver_pay        77
bank_transfer    74
card             70
Name: count, dtype: int64


## 컬럼 선택 - Series 와 DataFrame


In [6]:
# 컬럼 하나 -> Series (1차원)
city_series = customers["city"]

# 컬럼 여러 개(리스트) -> DataFrame (2차원)
customer_view = customers[["customer_id", "gender", "age", "city"]]

print(type(city_series))
print(type(customer_view))
print()
print(customer_view.head())


<class 'pandas.Series'>
<class 'pandas.DataFrame'>

   customer_id gender  age city
0            1      F   19   광주
1            2      F   32   대구
2            3      F   61   성남
3            4      F   55   울산
4            5      F   19   부산


## 단일 조건 필터링 - 불리언 마스크


In [7]:
# 조건식 자체는 True / False 목록(불리언 마스크)이다
mask = customers["age"] >= 30
print(type(mask))
print(mask.head())
print()

# 마스크를 대괄호에 넣어야 True인 행만 남는다
customers_over_30 = customers[mask]
print("전체:", len(customers), "/ 30세 이상:", len(customers_over_30))
print(customers_over_30.head())


<class 'pandas.Series'>
0    False
1     True
2     True
3     True
4    False
Name: age, dtype: bool

전체: 150 / 30세 이상: 111
   customer_id name gender  age city signup_date
1            2  김정호      F   32   대구  2026-01-03
2            3  이경수      F   61   성남  2024-08-13
3            4  조영호      F   55   울산  2026-06-14
5            6  김지원      F   32   성남  2026-08-28
6            7  이상현      F   53   인천  2025-02-12


## 복합 조건 - & | ~ (각 조건은 괄호로 감싼다)


In [8]:
# 30세 이상 "그리고" 서울 거주
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]
print("30세 이상 + 서울:", len(seoul_over_30))
print(seoul_over_30.head())


30세 이상 + 서울: 11
    customer_id name gender  age city signup_date
8             9  송지민      M   69   서울  2025-12-20
14           15  장정식      M   69   서울  2026-08-05
29           30  이민재      F   32   서울  2023-09-14
47           48  김예은      F   47   서울  2025-06-02
65           66  김재호      F   39   서울  2026-02-03


In [9]:
# 서울 "또는" 부산  -> isin() 이 읽기 쉽다
seoul_or_busan = customers[customers["city"].isin(["서울", "부산"])]
print(seoul_or_busan["city"].value_counts())


city
부산    16
서울    15
Name: count, dtype: int64


In [10]:
# 완료 주문이 "아닌" 주문  -> 틸드(~)
not_completed = orders[~(orders["order_status"] == "completed")]
print(not_completed["order_status"].value_counts(dropna=False))


order_status
cancelled    64
refunded     52
Name: count, dtype: int64


## 정렬 - sort_values()


In [11]:
# 가격이 높은 상품 10개
expensive = products.sort_values("price", ascending=False).head(10)
print(expensive[["product_id", "product_name", "category", "price"]])


    product_id product_name category   price
98          99    뷰티 상품 099       뷰티  200000
69          70    패션 상품 070       패션  198000
57          58    식품 상품 058       식품  197000
42          43    뷰티 상품 043       뷰티  197000
23          24   스포츠 상품 024      스포츠  196000
8            9   스포츠 상품 009      스포츠  193000
36          37    뷰티 상품 037       뷰티  193000
71          72    뷰티 상품 072       뷰티  189000
7            8   스포츠 상품 008      스포츠  189000
52          53  생활용품 상품 053     생활용품  188000


In [12]:
# 카테고리 안에서 가격이 높은 순서 (기준 2개, 방향 각각 지정)
by_cat = products.sort_values(["category", "price"], ascending=[True, False])
print(by_cat[["category", "product_name", "price"]].head(10))


   category product_name   price
39       도서    도서 상품 040  174000
31       도서    도서 상품 032  172000
68       도서    도서 상품 069  157000
6        도서    도서 상품 007  142000
35       도서    도서 상품 036  126000
93       도서    도서 상품 094  124000
62       도서    도서 상품 063  123000
72       도서    도서 상품 073  114000
99       도서    도서 상품 100  102000
16       도서    도서 상품 017   84000


In [13]:
print(customers["city"].unique())

<StringArray>
['광주', '대구', '성남', '울산', '부산', '인천', '수원', '서울', '대전', '고양']
Length: 10, dtype: str


In [14]:
# 서울, 부산, 인천 도시의 고객들은 누구인가?
city_customers = customers[customers["city"].isin(["서울", "부산", "인천"])]

print(city_customers.head(10))

    customer_id name gender  age city signup_date
4             5  이예원      F   19   부산  2024-11-14
6             7  이상현      F   53   인천  2025-02-12
8             9  송지민      M   69   서울  2025-12-20
14           15  장정식      M   69   서울  2026-08-05
15           16  강보람      M   52   부산  2024-09-23
18           19  김성수      M   54   인천  2024-10-28
19           20  백영호      F   20   인천  2025-01-09
20           21  박중수      M   23   인천  2026-06-03
29           30  이민재      F   32   서울  2023-09-14
39           40  박예준      M   23   서울  2024-08-03


In [15]:
print(city_customers["city"].value_counts())

city
부산    16
서울    15
인천    14
Name: count, dtype: int64


## 파생 컬럼 - line_total (주문상세 한 행의 금액)


In [16]:
# 원본을 지키기 위해 작업용 복사본을 만든다
order_items_work = order_items.copy()

order_items_work["line_total"] = (
    order_items_work["quantity"] * order_items_work["unit_price"]
)

print(order_items_work[["order_item_id", "order_id", "quantity", "unit_price", "line_total"]].head())


   order_item_id  order_id  quantity  unit_price  line_total
0              1         1         3      102000      306000
1              2         1         5       25000      125000
2              3         1         3      142000      426000
3              4         1         3      193000      579000
4              5         2         4      189000      756000


In [17]:
# 첫 행을 손으로 검산해서 계산식이 맞는지 확인한다
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected, "/ 파생 컬럼:", actual, "/ 일치:", expected == actual)


수작업: 306000 / 파생 컬럼: 306000 / 일치: True


## 주문 정보 병합 - order_sales


In [18]:
# 필요한 컬럼만 골라서 붙인다 (_x, _y 방지)
order_sales = order_items_work.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator="order_match",
)

print("병합 전:", len(order_items_work), "-> 병합 후:", len(order_sales))
print(order_sales["order_match"].value_counts(dropna=False))


병합 전: 764 -> 병합 후: 764
order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64


## 날짜형 변환 - to_datetime


In [19]:
order_sales_test = order_sales.copy()

print("변환 전 dtype:", order_sales_test["order_date"].dtype)

order_sales_test["order_date"] = pd.to_datetime(
    order_sales_test["order_date"],
    errors="coerce",
)

print("변환 후 dtype:", order_sales_test["order_date"].dtype)
print("order_date에 있는 결측치 수: ", order_sales_test["order_date"].isna().sum())


변환 전 dtype: str
변환 후 dtype: datetime64[us]
order_date에 있는 결측치 수:  0


## coerce 동작 확인 - 없는 날짜를 넣어보기


In [20]:
order_sales_test2 = order_sales.copy()

# 2026-02-30 은 존재하지 않는 날짜다
order_sales_test2.loc[0, "order_date"] = "2026-02-30"

order_sales_test2["order_date"] = pd.to_datetime(
    order_sales_test2["order_date"],
    errors="coerce",
)

print(order_sales_test2.info())
print(order_sales_test2.head(5))
print("order_date에 있는 결측치 수: ", order_sales_test2["order_date"].isna().sum())


<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_item_id  764 non-null    int64         
 1   order_id       764 non-null    int64         
 2   product_id     764 non-null    int64         
 3   quantity       764 non-null    int64         
 4   unit_price     764 non-null    int64         
 5   line_total     764 non-null    int64         
 6   customer_id    764 non-null    int64         
 7   order_date     763 non-null    datetime64[us]
 8   order_status   764 non-null    str           
 9   order_match    764 non-null    category      
dtypes: category(1), datetime64[us](1), int64(7), str(1)
memory usage: 54.7 KB
None
   order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0              1         1         100         3      102000      306000   
1              2         1          87         5       250

C:\Users\apll1\AppData\Local\Temp\ipykernel_14784\2453984350.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_sales_test2["order_date"] = pd.to_datetime(


In [21]:
order_sales_test2.head(5)

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match
0,1,1,100,3,102000,306000,123,NaT,completed,both
1,2,1,87,5,25000,125000,123,2026-07-08,completed,both
2,3,1,7,3,142000,426000,123,2026-07-08,completed,both
3,4,1,9,3,193000,579000,123,2026-07-08,completed,both
4,5,2,72,4,189000,756000,77,2025-09-23,cancelled,both


In [22]:
order_sales_test2 = order_sales_test2.dropna()
print(order_sales_test2["order_date"].isna().sum())

0


In [24]:
# 파생컬럼 order_month 추가하고 0으로 초기화
order_sales_test2["order_month"] = order_sales_test2["order_date"].dt.to_period("M").astype(str) # 연-월 형태로 변환이므로 str로 변환
print(order_sales_test2.head())

   order_item_id  order_id  product_id  quantity  unit_price  line_total  \
1              2         1          87         5       25000      125000   
2              3         1           7         3      142000      426000   
3              4         1           9         3      193000      579000   
4              5         2          72         4      189000      756000   
5              6         3          33         5      124000      620000   

   customer_id order_date order_status order_match order_month  
1          123 2026-07-08    completed        both     2026-07  
2          123 2026-07-08    completed        both     2026-07  
3          123 2026-07-08    completed        both     2026-07  
4           77 2025-09-23    cancelled        both     2025-09  
5          138 2026-01-20    cancelled        both     2026-01  
